# 🏥 Medical Report Generation - Transformer (Swin + BioGPT)
**Kaggle P100/T4 (16GB VRAM) üzerinde eğitim**

Repo: https://github.com/MustafaYagizKaragoz/medical-report-generation-cxr

## 1️⃣ Ortam Kurulumu

In [ ]:
# GPU kontrolü
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
# Gerekli paketleri yükle
!pip install -q transformers>=4.30.0 rouge-score pycocoevalcap

## 2️⃣ Repoyu İndir

In [ ]:
import os

# GitHub reposunu klonla
if not os.path.exists('/kaggle/working/medical-report-generation-cxr'):
    !git clone https://github.com/MustafaYagizKaragoz/medical-report-generation-cxr.git /kaggle/working/repo
else:
    print('Repo zaten mevcut')

os.chdir('/kaggle/working/repo')
print('Çalışma dizini:', os.getcwd())

## 3️⃣ Veri Setini Bağla

⚠️ **Önce yapman gereken:**
1. Kaggle'da **Datasets** sekmesine git
2. "New Dataset" → CSV dosyalarını yükle (`train_processed.csv`, `val_processed.csv`, `test_processed.csv`)
3. Görüntü dosyalarını da ayrı bir dataset olarak yükle
4. Bu notebook'a sağ taraftaki **"+ Add Data"** butonuyla ekle

In [ ]:
# Veri yollarını ayarla (Kaggle dataset yollarına göre güncelle)
# Kaggle'a yüklediğin dataset'in adına göre değiştir

TRAIN_CSV  = '/kaggle/input/mimic-cxr-processed/train_processed.csv'
VAL_CSV    = '/kaggle/input/mimic-cxr-processed/val_processed.csv'
TEST_CSV   = '/kaggle/input/mimic-cxr-processed/test_processed.csv'
IMAGE_DIR  = '/kaggle/input/mimic-cxr-images/files'  # Görüntü klasörü

# Dosyaların varlığını kontrol et
import os
for path in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    exists = os.path.exists(path)
    print(f'{'✅' if exists else '❌'} {path}')

## 4️⃣ Config Ayarları (Kaggle için optimize edilmiş)

In [ ]:
# Config'i Kaggle ortamına göre güncelle
import sys
sys.path.insert(0, '/kaggle/working/repo')

from config import Config

# Kaggle-spesifik path güncellemeleri
Config.TRAIN_PROCESSED_CSV = TRAIN_CSV
Config.VAL_PROCESSED_CSV   = VAL_CSV
Config.TEST_PROCESSED_CSV  = TEST_CSV
Config.IMAGE_DIR           = IMAGE_DIR
Config.TRANSFORMER_CHECKPOINT_DIR = '/kaggle/working/checkpoints_transformer'
Config.LOG_DIR             = '/kaggle/working/logs'

# Kaggle P100 (16GB) için optimize edilmiş ayarlar
Config.TRANSFORMER_BATCH_SIZE       = 2    # P100'de 2 batch güvenli
Config.TRANSFORMER_GRAD_ACCUM_STEPS = 8    # Efektif batch = 2x8 = 16
Config.TRANSFORMER_USE_AMP          = True # FP16 → %40 daha az bellek
Config.TRANSFORMER_MAX_LENGTH       = 128  # 150→128 biraz bellek tasarrufu
Config.TRANSFORMER_IMAGE_SIZE       = 384
Config.TRANSFORMER_LEARNING_RATE    = 5e-5
Config.TRANSFORMER_FINE_TUNE_START_EPOCH = 3
Config.NUM_WORKERS = 2  # Kaggle'da 2 worker çalışır
Config.EPOCHS = 20
Config.PATIENCE = 5

print('✅ Config güncellendi!')
print(f'   Batch size:        {Config.TRANSFORMER_BATCH_SIZE}')
print(f'   Grad accumulation: {Config.TRANSFORMER_GRAD_ACCUM_STEPS}')
print(f'   Efektif batch:     {Config.TRANSFORMER_BATCH_SIZE * Config.TRANSFORMER_GRAD_ACCUM_STEPS}')
print(f'   AMP (FP16):        {Config.TRANSFORMER_USE_AMP}')

## 5️⃣ Modeli Başlat ve Eğit

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Eğitimi başlat
# (train_transformer.py'nin main() fonksiyonunu çağırır)

from src.data_loader.data_transformer import get_transformer_dataloaders
from src.models.transformer_model import MedicalTransformer

# DataLoader'ları oluştur
Config.setup()
print('\n📥 Veri yükleniyor...')
train_loader, val_loader, tokenizer = get_transformer_dataloaders(
    train_csv=Config.TRAIN_PROCESSED_CSV,
    val_csv=Config.VAL_PROCESSED_CSV,
    image_dir=Config.IMAGE_DIR,
    batch_size=Config.TRANSFORMER_BATCH_SIZE,
    max_length=Config.TRANSFORMER_MAX_LENGTH,
    image_size=Config.TRANSFORMER_IMAGE_SIZE,
    num_workers=Config.NUM_WORKERS
)
print(f'✅ Train: {len(train_loader)} batch | Val: {len(val_loader)} batch')

In [ ]:
# Modeli başlat
model = MedicalTransformer(
    encoder_name=Config.TRANSFORMER_ENCODER,
    decoder_name=Config.TRANSFORMER_DECODER,
    freeze_encoder=True,
    max_length=Config.TRANSFORMER_MAX_LENGTH,
    num_beams=Config.TRANSFORMER_NUM_BEAMS
).to(Config.DEVICE)

print('✅ Model hazır!')

In [ ]:
# Eğitim döngüsünü çalıştır
import train_transformer
train_transformer.main()

## 6️⃣ Checkpoint'leri Kaydet (İndirmek İçin)

In [ ]:
# En iyi modeli zip'le ve kaydet
import shutil

shutil.make_archive(
    '/kaggle/working/best_model_transformer',
    'zip',
    '/kaggle/working/checkpoints_transformer/best_model'
)
print('✅ Model zip olarak kaydedildi: /kaggle/working/best_model_transformer.zip')
print('   Sağ taraftaki Output sekmesinden indirebilirsin.')